In [1]:
%load_ext autoreload
%autoreload 2
from experiment import find_all_datasets, find_all_experiments

In [2]:
dses = find_all_datasets("../../datasets/")
imagenet = dses["imagenet"]
split = "trainUval"

Found dataset: cifar with splits: ['train', 'trainUval', 'val']
Found dataset: emnist with splits: ['train', 'trainUval', 'val']
Adjusted labels for dataset emnist, split train to be zero-indexed.
Adjusted labels for dataset emnist, split trainUval to be zero-indexed.
Adjusted labels for dataset emnist, split val to be zero-indexed.
Found dataset: emnist_balanced with splits: ['train', 'trainUval', 'val']
Adjusted labels for dataset emnist_balanced, split train to be zero-indexed.
Adjusted labels for dataset emnist_balanced, split trainUval to be zero-indexed.
Adjusted labels for dataset emnist_balanced, split val to be zero-indexed.
Found dataset: emnist_letters with splits: ['train', 'trainUval', 'val']
Adjusted labels for dataset emnist_letters, split train to be zero-indexed.
Adjusted labels for dataset emnist_letters, split trainUval to be zero-indexed.
Adjusted labels for dataset emnist_letters, split val to be zero-indexed.
Found dataset: imagenet with splits: ['train', 'trainUv

In [3]:
data_dir = "C:/home/eurovis_data/landscape_data_pt/"
strees_dir = "C:/home/eurovis_data/strees_pt/"

In [4]:
exps = find_all_experiments(dses, data_dir, strees_dir)
exp = exps[1]
assert exp.split == split

Model: resnet50pt, Dataset: imagenet, Split: train
Model: resnet50pt, Dataset: imagenet, Split: trainUval
Model: resnet50pt, Dataset: imagenet, Split: val


In [5]:
from basic_utils import get_labels, get_partition, get_tree, get_order_and_weights
labels = get_labels(exp)
partition = get_partition(exp)

assert len(partition) == len(labels)

In [6]:
import pyct as ct
import numpy as np
import pickle as pkl

data, _ = get_tree(exp)
simpl = ct.SimplifyCT() # type: ignore
simpl.setInput(data)

order, wts = get_order_and_weights(exp)

with open("imagenet_pt_homo_valley_plot_coverages.pkl", "rb") as f:
    res = pkl.load(f)

In [16]:
props = [float(x) for x in res.keys()]
props

[0.5,
 0.5555555555555556,
 0.6111111111111112,
 0.6666666666666666,
 0.7222222222222222,
 0.7777777777777778,
 0.8333333333333333,
 0.8888888888888888,
 0.9444444444444444,
 1.0]

In [8]:
prop = props[9]
results = res[prop]

fns, remaining_homo, maj_class_homo_cov, maj_class_homo_counts, class_homo_covs, class_all_covs = results

In [9]:
import plotly.express as px

px.line(x=fns, y=remaining_homo, labels={"x": "Function Value", "y": "Number of Homogeneous Valleys"}, title=f"Homogeneous Valleys vs Function Value (Homogeneity Threshold = {prop})")

In [ ]:
classes_needed = 0.91 * 1000 # 1000 classes, the following criteria must apply to at least 910 of them
maj_coverage_needed = 0.05 # at least 5% coverage in homogeneous valleys where they are the only class (100% proportion in valley) 
total_coverage_needed = 0.99 # at least 99% total coverage in all valleys

represented = [[mc >= maj_coverage_needed and tc >= total_coverage_needed for mc, tc in zip(maj_cov, total_cov)].count(True) >= classes_needed 
            	for maj_cov, total_cov in zip(maj_class_homo_cov, class_all_covs)]

# find first index where this fails to be true
first_idx = represented.index(False)
last_idx = len(fns) - represented[::-1].index(True) - 1

# sanity check, once this homogeneity is destroyed, we never recover it
assert last_idx == first_idx - 1


first_idx, last_idx, len(fns), fns[first_idx - 1], list(zip(maj_class_homo_cov[first_idx - 1], class_all_covs[first_idx - 1], maj_class_homo_counts[first_idx - 1]))

(583,
 582,
 9942,
 1.1920923270736239e-07,
 [(0.6577777777777778, 1.1703703703703703, 13),
  (0.5259259259259259, 1.1244444444444444, 11),
  (0.25037037037037035, 1.1481481481481481, 12),
  (0.4162962962962963, 1.1481481481481481, 7),
  (0.31407407407407406, 1.1407407407407408, 9),
  (0.27111111111111114, 1.125925925925926, 11),
  (0.4688888888888889, 1.3185185185185184, 24),
  (0.4148148148148148, 1.1925925925925926, 10),
  (0.3503703703703704, 1.2814814814814814, 22),
  (0.016296296296296295, 1.0192592592592593, 2),
  (0.8725925925925926, 1.2148148148148148, 9),
  (0.040740740740740744, 1.2296296296296296, 5),
  (0.7785185185185185, 1.2222222222222223, 8),
  (0.08962962962962963, 1.1037037037037036, 7),
  (0.9207407407407407, 1.162962962962963, 14),
  (0.03259259259259259, 1.0429629629629629, 3),
  (0.7288888888888889, 1.1185185185185185, 5),
  (0.6896296296296296, 1.2666666666666666, 2),
  (0.10666666666666667, 1.1037037037037036, 8),
  (0.9822222222222222, 1.2066666666666668, 10),